# Capítulo 4 · Un contrato de datos que cambia lo que estimamos

Una agencia de empleo recibe dos análisis de la **misma tabla**: uno dice que un año más de escolaridad sube el salario en 1, el otro dice 1.5. Ninguno tiene errores de código.

**La pregunta de hoy:** si los datos son los mismos, ¿de dónde sale la diferencia, y cómo evitamos que vuelva a pasar?

**En esta sesión vamos a:**

1. Encontrar el origen del desacuerdo.
2. Escribir los roles de cada columna en un contrato único: el **esquema**.
3. Ver qué errores atrapa ese contrato y cuál no puede atrapar.
4. Aplicarlo a datos reales de capacitación laboral.

> *Nota:* la Parte I usa datos sintéticos, con efecto conocido. La Parte III usa `jobs/`, donde el efecto verdadero se desconoce.

# Parte I · Dos análisis, una tabla

Simulamos un mundo donde conocemos la verdad:

| Variable | Rol | Ecuación |
|:--|:--|:--|
| $P$ educación de los padres | covariable | $P \sim N(0,1)$ |
| $E$ edad | covariable | $E \sim N(0,1)$ |
| $D$ escolaridad | **tratamiento** | $D = 0.5P + \varepsilon_D$ |
| $Y$ salario | **resultado** | $Y = D + 0.8P + 0.5E + \varepsilon_Y$ |
| $S$ experiencia | ¿covariable? | $S = E - D$ |

Los términos $\varepsilon_Y$ son ruido aleatorio: todo lo que influye y no modelamos.

Efecto verdadero de un año de escolaridad: **1**. La experiencia se calcula como edad menos escolaridad, una práctica habitual en datos laborales.

**Antes de ejecutar:** el análisis A controla por padres y edad; el B, por padres y experiencia. ¿Darán lo mismo?

In [22]:
import json
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd

from lib.data_io import load_scenario
from lib.identification import ols_ate

In [34]:
EFECTO = 1.0      # efecto verdadero de la escolaridad
BETA_EDAD = 0.5   # cuánto sube el salario por cada unidad de edad

g = np.random.default_rng(4)
n = 5000
P, E, ruido_D, ruido_Y = g.normal(size=(4, n))

D = 0.5 * P + ruido_D                          # escolaridad
Y = EFECTO * D + 0.8 * P + BETA_EDAD * E + ruido_Y   # salario
S = E - D                                      # experiencia

datos = pd.DataFrame({"padres": P, "edad": E, "educacion": D,
                      "salario": Y, "experiencia": S})

a = ols_ate(datos.salario, datos.educacion, datos[["padres", "edad"]])
b = ols_ate(datos.salario, datos.educacion, datos[["padres", "experiencia"]])

print(f"A · padres y edad:         {a:.3f}")
print(f"B · padres y experiencia:  {b:.3f}")

A · padres y edad:         1.010
B · padres y experiencia:  1.518


**Qué pasó:** que dos análisis den números distintos no es lo normal. Si ambos conjuntos de covariables fueran válidos, los dos darían 1. Que difieran significa que uno está sesgado.

**El sesgado es B.** La experiencia lleva la escolaridad dentro: entre dos personas con la misma experiencia, la que estudió un año más es también un año mayor, y gana 0.5 más solo por eso. El análisis B se lo atribuye a la escuela: $1 + 0.5 = 1.5$.

En el grafo, la experiencia es un **collider**, es decir, una variable a la que llegan dos flechas: $D \to S \leftarrow E \to Y$. Como en el capítulo 3, condicionar en un collider abre un camino que estaba cerrado.

**Lo importante: B no cometió un error de código.** Siguió una práctica estándar: la ecuación de Mincer, base de la economía laboral, controla por experiencia. Lo que faltó fue una regla escrita que dijera "para esta pregunta, la experiencia no se controla". Esa regla es lo que construimos en la Parte II.

**Para discutir:**

1. Controla solo por `padres`. ¿Qué da? ¿Por qué coincide con A?
2. Cambia `n` a 50 000. ¿Desaparece la diferencia? ¿Por qué más datos no la arreglan?
3. Pon `BETA_EDAD = 0`. ¿Qué pasa con el análisis B?

# Parte II · El esquema como contrato

Los dos análisis discreparon porque cada uno escribió su propia lista de covariables. La solución: declarar los roles **una sola vez**, en un esquema que viaja con los datos y que todos los análisis consumen.

| Campo | Qué declara |
|:--|:--|
| `tratamiento` | la variable cuyo efecto queremos medir |
| `resultado` | la variable afectada |
| `covariables` | las que se pueden controlar sin sesgar |
| `posteriores` | las que el equipo sabe que se midieron después del tratamiento |

**Clave:** declarar una covariable es una **afirmación causal**, no técnica. El esquema puede revisar nombres; no puede comprobar que esa afirmación sea cierta.

La celda siguiente ya viene escrita: es el contrato en código.

In [35]:
@dataclass(frozen=True)
class EsquemaCausal:
    """Contrato de roles causales: se declara una vez y no se modifica."""
    tratamiento: str
    resultado: str
    covariables: tuple
    posteriores: tuple = ()   # medidas DESPUÉS del tratamiento (lo declara el equipo)

    def __post_init__(self):
        object.__setattr__(self, "covariables", tuple(self.covariables))
        object.__setattr__(self, "posteriores", tuple(self.posteriores))
        if not self.covariables:
            raise ValueError("Se necesita al menos una covariable")
        roles = self.columnas()
        dobles = sorted({c for c in roles if roles.count(c) > 1})
        if dobles:
            raise ValueError(f"Columnas con dos roles: {dobles}")
        tardias = sorted(set(self.covariables) & set(self.posteriores))
        if tardias:
            raise ValueError(f"Covariables medidas después del tratamiento: {tardias}")

    def columnas(self):
        return [self.tratamiento, self.resultado, *self.covariables]

    def validar(self, df):
        ausentes = sorted(set(self.columnas()) - set(df.columns))
        if ausentes:
            raise ValueError(f"Columnas ausentes en los datos: {ausentes}")
        if df[self.tratamiento].nunique() < 2:
            raise ValueError(f"El tratamiento '{self.tratamiento}' no varía en estos datos")

    def proyectar(self, df):
        """Devuelve resultado, tratamiento y covariables, en el orden declarado."""
        self.validar(df)
        return df[self.resultado], df[self.tratamiento], df[list(self.covariables)]

    def a_json(self):
        return json.dumps({"tratamiento": self.tratamiento, "resultado": self.resultado,
                           "covariables": list(self.covariables),
                           "posteriores": list(self.posteriores)}, ensure_ascii=False)

    @classmethod
    def desde_json(cls, texto):
        r = json.loads(texto)
        return cls(r["tratamiento"], r["resultado"], r["covariables"],
                   r.get("posteriores", []))

## Declarar, guardar y recuperar

El esquema se escribe como texto (aquí, JSON) y viaja junto al archivo de datos. Quien lo lee obtiene exactamente los mismos roles, y por tanto exactamente la misma estimación.

**Antes de ejecutar:** ¿qué estimación debería dar el esquema que declara padres y edad como covariables?

In [37]:
esquema = EsquemaCausal(tratamiento="educacion", resultado="salario",
                        covariables=("padres", "edad"))

texto = esquema.a_json()                          # esto se guarda junto a los datos
print("Lo que viaja con los datos:\n", texto)

otro_analista = EsquemaCausal.desde_json(texto)   # otra persona lo lee
y, t, X = otro_analista.proyectar(datos)
print("\nEstimación del otro analista:", round(ols_ate(y, t, X), 3))

Lo que viaja con los datos:
 {"tratamiento": "educacion", "resultado": "salario", "covariables": ["padres", "edad"], "posteriores": []}

Estimación del otro analista: 1.01


**Para discutir:** invierte el orden físico de las columnas de `datos` y vuelve a proyectar. ¿Cambia el orden de $X$? ¿Por qué conviene que ese orden lo fije el esquema y no la tabla?

## Qué errores atrapa el contrato

Cada caso de abajo es un error real de operación. Provocamos los seis a propósito y vemos si el contrato los rechaza.

El contrato tiene dos momentos de control, y conviene distinguirlos: unos errores saltan **al declarar** el esquema, porque se contradice a sí mismo, y otros **al validarlo contra la tabla**, porque el esquema está bien pero los datos no son los que esperaba.

**Antes de ejecutar:** para cada caso, ¿en cuál de los dos momentos debería saltar?

In [36]:
casos = [
    ("el origen renombró una columna",          datos.rename(columns={"edad": "edad_nueva"}), {}),
    ("un filtro dejó el tratamiento constante", datos.assign(educacion=0),                    {}),
    ("el tratamiento entró como covariable",    datos, {"covariables": ("padres", "educacion")}),
    ("el resultado entró como covariable",      datos, {"covariables": ("padres", "salario")}),
    ("la lista de covariables quedó vacía",     datos, {"covariables": ()}),
]

for nombre, tabla, cambio in casos:
    try:
        contrato = replace(esquema, **cambio)     # 1. declarar
    except ValueError as error:
        print(f"RECHAZADO al declarar · {nombre}\n     → {error}")
        continue
    try:
        contrato.validar(tabla)                   # 2. validar contra la tabla
    except ValueError as error:
        print(f"RECHAZADO al validar  · {nombre}\n     → {error}")
    else:
        print(f"ACEPTADO              · {nombre}  ← revisar este caso")

RECHAZADO al validar  · el origen renombró una columna
     → Columnas ausentes en los datos: ['edad']
RECHAZADO al validar  · un filtro dejó el tratamiento constante
     → El tratamiento 'educacion' no varía en estos datos
RECHAZADO al declarar · el tratamiento entró como covariable
     → Columnas con dos roles: ['educacion']
RECHAZADO al declarar · el resultado entró como covariable
     → Columnas con dos roles: ['salario']
RECHAZADO al declarar · la lista de covariables quedó vacía
     → Se necesita al menos una covariable


## Y qué error NO puede atrapar

El esquema con la experiencia como covariable no tiene ninguna contradicción: las columnas existen, ningún rol se repite y el tratamiento varía. Pasa la validación y entrega un número distinto.

In [27]:
esquema_alternativo = replace(esquema, covariables=("padres", "experiencia"))

for nombre, contrato in [("original", esquema), ("alternativo", esquema_alternativo)]:
    y, t, X = contrato.proyectar(datos)   # ninguno lanza error
    print(f"{nombre:12} {str(contrato.covariables):32} → {round(ols_ate(y, t, X), 3)}")

print("\nEl esquema original no cambió:", esquema.covariables)

original     ('padres', 'edad')               → 1.01
alternativo  ('padres', 'experiencia')        → 1.518

El esquema original no cambió: ('padres', 'edad')


**Qué nos dice esto:** los dos esquemas son válidos, pero solo uno es defendible en este mundo. El contrato revisa **nombres**, no causalidad.

**Entonces, ¿para qué sirve el contrato?** Para que la discusión sobre si la experiencia pertenece a las covariables ocurra **en un solo lugar y quede registrada**, en vez de resolverse distinto en cada análisis sin que nadie lo note.

# Parte III · La pregunta que sí decide: ¿cuándo se midió?

El contrato no sabe causalidad, pero sí puede registrar algo que el equipo conoce y que decide casi todos los casos: **si la variable existía antes de asignar el tratamiento**.

Con los datos de la carpeta `jobs` fabricamos una variable tentadora: `trabajo_78`, que vale 1 si la persona tuvo algún ingreso en 1978. Predice el ingreso casi a la perfección, pero se construye **con el resultado**.

**Antes de ejecutar:** ¿en qué dirección esperas que mueva la estimación incluirla?

In [38]:
from pathlib import Path

CARPETA = Path("jobs")                 # la carpeta que pusiste al lado del notebook
TRATAMIENTO, RESULTADO = "treatment", "re78"

tabla = pd.read_csv(CARPETA / "observed_train.csv")
print("Columnas del archivo:", list(tabla.columns))

faltan = {TRATAMIENTO, RESULTADO} - set(tabla.columns)
assert not faltan, f"Estas columnas tienen otro nombre en el archivo: {faltan}"

# Todo lo demás son covariables, salvo identificadores y columnas de verdad oculta
COVARIABLES = tuple(c for c in tabla.columns
                    if c not in (TRATAMIENTO, RESULTADO)
                    and not c.startswith(("truth_", "row_id", "id")))

# trabajo_78: tentadora porque predice muy bien, pero se construye CON el resultado
tabla["trabajo_78"] = (tabla[RESULTADO] > 0).astype(int)

esquema_jobs = EsquemaCausal(tratamiento=TRATAMIENTO, resultado=RESULTADO,
                             covariables=COVARIABLES, posteriores=("trabajo_78",))
esquema_jobs.validar(tabla)

print("\nRoles declarados")
print(f"  tratamiento : {esquema_jobs.tratamiento}  (participar en el programa)")
print(f"  resultado   : {esquema_jobs.resultado}  (ingreso de 1978, en dólares)")
print(f"  covariables : {', '.join(esquema_jobs.covariables)}")
print(f"  posteriores : {', '.join(esquema_jobs.posteriores)}  ← declarada, y fuera de las covariables")
print(f"\nValidado: {len(tabla)} filas, tratamiento con "
      f"{tabla[esquema_jobs.tratamiento].nunique()} valores distintos.")

Columnas del archivo: ['row_id', 'age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75', 'treatment', 're78']

Roles declarados
  tratamiento : treatment  (participar en el programa)
  resultado   : re78  (ingreso de 1978, en dólares)
  covariables : age, educ, black, hispan, married, nodegree, re74, re75
  posteriores : trabajo_78  ← declarada, y fuera de las covariables

Validado: 1033 filas, tratamiento con 2 valores distintos.


In [32]:
# Intentar declararla como covariable: el contrato ya sabe que es posterior
try:
    replace(esquema_jobs, covariables=esquema_jobs.covariables + ("trabajo_78",))
except ValueError as error:
    print("RECHAZADO →", error)

RECHAZADO → Covariables medidas después del tratamiento: ['trabajo_78']


In [33]:
# Solo para ver el daño: así habría quedado si nadie la hubiera declarado
y, t, X = esquema_jobs.proyectar(tabla)
X_con_fuga = tabla[list(esquema_jobs.covariables) + ["trabajo_78"]]

comparacion = pd.Series({
    "ajuste declarado (sin trabajo_78)": ols_ate(y, t, X),
    "incluyendo una variable posterior": ols_ate(y, t, X_con_fuga),
})
print("Efecto estimado del programa sobre re78, en dólares:\n")
print(comparacion.round(0).to_string())

Efecto estimado del programa sobre re78, en dólares:

ajuste declarado (sin trabajo_78)    1596.0
incluyendo una variable posterior    1147.0


**Para discutir:**

1. `trabajo_78` predice el ingreso muy bien. ¿Por qué eso no la legitima como control?
2. El contrato la rechazó porque **alguien la declaró** como posterior. ¿Podría haberlo averiguado solo, mirando la tabla?
3. ¿Quién, en tu equipo, firmaría esa declaración?

*Idea clave:* la auditoría usa conocimiento del proceso, no de los datos. El contrato guarda esa declaración y la vuelve ejecutable; no la descubre.

## Cierre: la ficha que acompaña a la estimación

Completa estas cuatro líneas:

1. **Roles:** tratamiento, resultado y covariables declaradas.
2. **Variable excluida y motivo:** cuál dejaste fuera y por qué.
3. **Qué rechaza el contrato:** los errores de nombres y estructura que ya no pueden pasar.
4. **Qué no rechaza:** una covariable mal elegida que nadie declaró como posterior.

**Recuerda:** el mismo esquema que declaraste aquí es el que después registra la ficha del modelo y el que el servicio exige en cada consulta. Por eso se escribe una vez, se versiona y viaja con los datos.